Intialization

In [0]:
import pyspark.sql.functions as F
from pyspark.sql.types import StringType
from pyspark.sql.functions import col, trim

Reading from Bronze table

In [0]:
df= spark.table("workspace.bronze.cust_info")

Doing transformations

In [0]:
df.display()

Trimming

In [0]:
for field in df.schema.fields:
    if isinstance(field.dataType, StringType):
        df = df.withColumn(field.name, trim(col(field.name)))


Normalization, abbrevations

In [0]:
df = (
    df
    .withColumn(
        "cst_marital_status",
        F.when(F.upper(F.col("cst_marital_status")) == "S", "Single")
         .when(F.upper(F.col("cst_marital_status")) == "M", "Married")
         .otherwise("n/a")
    )
    .withColumn(
        "cst_gndr",
        F.when(F.upper(F.col("cst_gndr")) == "F", "Female")
         .when(F.upper(F.col("cst_gndr")) == "M", "Male")
         .otherwise("n/a")
    )
)
    

In [0]:
df.select("cst_marital_status").distinct().show(50, False)
#This will show the actual values in the column. They might contain unexpected values, spaces, nulls, or different codes.

Remove Records with Missing Customer ID

In [0]:
df=df.filter(col("cst_id").isNotNull())

Renaming Columns

In [0]:
RENAME_MAP={
    "cst_id":"customer_id",
    "cst_key":"customer_key",
    "cst_firstname":"customer_firstname",
    "cst_lastname":"customer_lastname",
    "cst_marital_status":"customer_marital_status",
    "cst_gndr":"customer_gender",
    "cst_create_date":"customer_create_date"
}
for old,new in RENAME_MAP.items():
    df = df.withColumnRenamed(old,new)

Sanity checks of dataframe

In [0]:
df.limit(10).display()

In [0]:
df.display()

Write in to the Silver table.

In [0]:
df.write.mode("overwrite").format("delta").saveAsTable("workspace.silver.crm_customers")